# Pegangan UTS Data Mining II — Audio Signal

**Cara pakai: import → konfigurasi bersama → sisakan SATU chunk loading → analisis.**

Hapus judul dan semua cell kode dari chunk loading yang tidak dipakai, lalu baru Run All.
Lokasi `/content/...` adalah contoh Colab; ganti sesuai komputer. Tidak ada `SOURCE`.
Notebook belum dijalankan pada dataset.

| Chunk | Input | Lanjut ke |
|---|---|---|
| A | Folder audio lokal | BAGIAN AUDIO MENTAH |
| B | ZIP audio | BAGIAN AUDIO MENTAH |
| C | CSV/Excel lokasi audio | BAGIAN AUDIO MENTAH |
| D | Dataset audio dari Kaggle | BAGIAN AUDIO MENTAH |
| E | CSV/Excel fitur yang sudah dihitung | STANDARDISASI BERSAMA |

A–D menghasilkan `df_files`; jalankan metadata, waveform, spectrogram, dan ekstraksi fitur.
E menghasilkan `df_features` langsung. **Jika memilih E, hapus juga seluruh BAGIAN AUDIO
MENTAH**, karena tabelnya sudah berisi fitur, bukan sinyal. Semua jalur bertemu di
STANDARDISASI BERSAMA untuk clustering dan evaluasi.


## 1. Import umum
Install package sekali bila diperlukan. Package Kagglehub berada pada chunk Kaggle.


In [ ]:
# Jalankan hanya jika diperlukan:
# %pip install numpy pandas matplotlib scipy scikit-learn librosa soundfile openpyxl


In [ ]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
from scipy.signal import spectrogram
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score


## 2. Konfigurasi preprocessing dan analisis bersama

Sample rate fitur disamakan. Sesuaikan `TARGET_SR` dengan domain dan soal. Trim, pre-emphasis,
dan normalisasi amplitudo dapat mengubah karakter sinyal. Pre-emphasis dan normalisasi
amplitudo dibiarkan nonaktif. `MAX_SECONDS=None` membaca seluruh durasi.
Pengaturan sinyal tidak digunakan jika memilih tabel fitur siap pakai.


In [ ]:
MAX_ITEMS = None
RANDOM_STATE = 42
PREVIEW_INDEX = 0
TARGET_SR = 22050
TRIM_SILENCE = True
PREEMPHASIS = False
NORMALIZE_AUDIO = False
MAX_SECONDS = None
MIN_SECONDS = 0.05
N_FFT = 2048
HOP_LENGTH = 512
N_MFCC = 13

K_FIXED = None
K_CANDIDATES = [2, 3, 4, 5, 6, 7, 8, 10]
SILHOUETTE_SAMPLE = 1000
USE_MINIBATCH = False


# PILIH SATU CHUNK LOADING — hapus alternatif lainnya


## CHUNK A — Folder audio lokal

Pencarian termasuk subfolder. Aktifkan label folder hanya jika subfolder pertama memang kategori audio.


In [ ]:
DATA_DIR = Path("/content/data/audio")
LABEL_FROM_FOLDER = False


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_DIR}")
extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aiff", ".aif"}
paths = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
labels = []
for file in paths:
    parts = file.relative_to(DATA_DIR).parts
    labels.append(parts[0] if LABEL_FROM_FOLDER and len(parts) > 1 else None)
df_files = pd.DataFrame({"path": paths, "label": labels})
failed = []
print("Audio ditemukan:", len(df_files))
display(df_files.head())


## CHUNK B — ZIP audio

Jika ada folder pembungkus, isi `AUDIO_SUBFOLDER`. Gunakan folder ekstraksi berbeda jika mengganti ZIP.


In [ ]:
ZIP_PATH = Path("/content/audio.zip")
EXTRACT_DIR = Path("/content/audio_extracted")
AUDIO_SUBFOLDER = ""  # contoh: "dataset/set_b"
LABEL_FROM_FOLDER = False

with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(EXTRACT_DIR)
print("Isi hasil ekstraksi:", [p.name for p in EXTRACT_DIR.iterdir()][:20])
DATA_DIR = EXTRACT_DIR / AUDIO_SUBFOLDER


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_DIR}")
extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aiff", ".aif"}
paths = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
labels = []
for file in paths:
    parts = file.relative_to(DATA_DIR).parts
    labels.append(parts[0] if LABEL_FROM_FOLDER and len(parts) > 1 else None)
df_files = pd.DataFrame({"path": paths, "label": labels})
failed = []
print("Audio ditemukan:", len(df_files))
display(df_files.head())


## CHUNK C — CSV/Excel lokasi audio

Tabel berisi path file audio. Path relatif dihitung dari folder tabel atau `FILE_BASE_DIR`. Label dipakai hanya untuk evaluasi.


In [ ]:
TABLE_PATH = Path("/content/audio_paths.csv")
PATH_COLUMN = "filename"
LABEL_COLUMN = None
FILE_BASE_DIR = None
CSV_SEP = ","
CSV_ENCODING = "utf-8-sig"

if TABLE_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_table = pd.read_excel(TABLE_PATH)
else:
    df_table = pd.read_csv(TABLE_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
display(df_table.head())
if PATH_COLUMN not in df_table.columns:
    raise ValueError(f"Periksa PATH_COLUMN. Kolom: {list(df_table.columns)}")
if LABEL_COLUMN is not None and LABEL_COLUMN not in df_table.columns:
    raise ValueError("LABEL_COLUMN tidak ditemukan.")


In [ ]:
missing = df_table[PATH_COLUMN].isna() | df_table[PATH_COLUMN].astype(str).str.strip().eq("")
failed = [{"path": f"row_{i}", "error": "Lokasi audio kosong"} for i in df_table.index[missing]]
df_table = df_table.loc[~missing].copy()
base_dir = TABLE_PATH.parent if FILE_BASE_DIR is None else Path(FILE_BASE_DIR)
paths = [Path(str(value).strip()) for value in df_table[PATH_COLUMN]]
paths = [p if p.is_absolute() else base_dir / p for p in paths]
labels = df_table[LABEL_COLUMN].to_numpy() if LABEL_COLUMN is not None else [None] * len(paths)
df_files = pd.DataFrame({"path": paths, "label": labels})
print("Baris tanpa lokasi dibuang:", missing.sum())
display(df_files.head())


## CHUNK D — Dataset audio dari Kaggle

Contoh memakai dataset heartbeat dari notebook awalmu. Ganti dataset dan subfolder sesuai soal. Download menghasilkan folder, kemudian file audio dicari dari folder tersebut.


In [ ]:
# %pip install kagglehub
import kagglehub

KAGGLE_DATASET = "kinguistics/heartbeat-sounds"
AUDIO_SUBFOLDER = "set_b"
LABEL_FROM_FOLDER = False

dataset_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("Lokasi dataset:", dataset_path)
print("Isi dataset:", [p.name for p in dataset_path.iterdir()][:20])
DATA_DIR = dataset_path / AUDIO_SUBFOLDER


In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Folder tidak ditemukan: {DATA_DIR}")
extensions = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aiff", ".aif"}
paths = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in extensions)
labels = []
for file in paths:
    parts = file.relative_to(DATA_DIR).parts
    labels.append(parts[0] if LABEL_FROM_FOLDER and len(parts) > 1 else None)
df_files = pd.DataFrame({"path": paths, "label": labels})
failed = []
print("Audio ditemukan:", len(df_files))
display(df_files.head())


## CHUNK E — CSV/Excel fitur audio siap pakai

**Khusus tabel fitur:** isi `FEATURE_COLUMNS`, hapus BAGIAN AUDIO MENTAH, lalu langsung ke STANDARDISASI BERSAMA. Kolom ID/label tidak masuk fitur. File audio asli tidak diperlukan.


In [ ]:
TABLE_PATH = Path("/content/audio_features.csv")
FEATURE_COLUMNS = ["mfcc_1", "mfcc_2", "rms"]  # ganti sesuai nama fitur yang diberikan
LABEL_COLUMN = None
CSV_SEP = ","
CSV_ENCODING = "utf-8-sig"

if TABLE_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df_table = pd.read_excel(TABLE_PATH)
else:
    df_table = pd.read_csv(TABLE_PATH, sep=CSV_SEP, encoding=CSV_ENCODING)
display(df_table.head())
if LABEL_COLUMN is not None and LABEL_COLUMN not in df_table.columns:
    raise ValueError("LABEL_COLUMN tidak ditemukan.")
if not FEATURE_COLUMNS or not set(FEATURE_COLUMNS).issubset(df_table.columns):
    raise ValueError(f"Periksa FEATURE_COLUMNS. Kolom: {list(df_table.columns)}")
if LABEL_COLUMN in FEATURE_COLUMNS or "cluster" in FEATURE_COLUMNS:
    raise ValueError("Label/cluster tidak boleh masuk FEATURE_COLUMNS.")


In [ ]:
if MAX_ITEMS is not None and len(df_table) > MAX_ITEMS:
    print("Sampling fitur:", MAX_ITEMS, "dari", len(df_table))
    df_table = df_table.sample(n=MAX_ITEMS, random_state=RANDOM_STATE)
df_features = df_table[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce").copy()
df_features["path"] = [f"row_{i}" for i in df_table.index]
df_features["label"] = df_table[LABEL_COLUMN] if LABEL_COLUMN is not None else None
feature_columns = FEATURE_COLUMNS.copy()
failed = []


# MULAI BAGIAN AUDIO MENTAH — hanya untuk chunk A/B/C/D

**Jika memilih chunk E (CSV fitur), hapus bagian ini sampai sebelum STANDARDISASI BERSAMA.**
Untuk A/B/C/D, pertahankan semua cell di bagian ini. Codec MP3/M4A bergantung pada environment;
file gagal akan dicatat. Sinyal dibaca sebagai array float, sehingga tidak perlu mengasumsikan
semua WAV merupakan integer PCM 16-bit.


In [ ]:
if df_files.empty:
    raise ValueError("Tidak ada audio. Periksa lokasi folder/file dan ekstensi.")
total_files = len(df_files)
if MAX_ITEMS is not None and len(df_files) > MAX_ITEMS:
    df_files = df_files.sample(n=MAX_ITEMS, random_state=RANDOM_STATE)
df_files = df_files.reset_index(drop=True)
print("File diproses:", len(df_files), "dari", total_files)


## 4. Pilih audio dan baca array

`librosa.load(..., sr=None, mono=True)` membaca array float dan mempertahankan sample rate
asli untuk preview. Stereo dirata-ratakan menjadi mono. Jika file pilihan gagal/kosong/senyap,
cell mencari file berikutnya. Preview tidak memakai trim atau pre-emphasis.


In [ ]:
if not 0 <= PREVIEW_INDEX < len(df_files):
    raise ValueError("PREVIEW_INDEX di luar jumlah file.")
signal = None
for index in np.roll(np.arange(len(df_files)), -PREVIEW_INDEX):
    preview_path = df_files.loc[index, "path"]
    try:
        y_preview, sr_preview = librosa.load(str(preview_path), sr=None, mono=True, duration=MAX_SECONDS)
        if len(y_preview) < 2 or not np.isfinite(y_preview).all() or np.max(np.abs(y_preview)) == 0:
            continue
        signal = y_preview
        break
    except Exception as error:
        print("Preview dilewati:", Path(preview_path).name, "|", str(error)[:100])
if signal is None:
    raise ValueError("Tidak ada audio yang dapat dibaca untuk preview.")

print("File:", preview_path)
print("Sample rate asli:", sr_preview, "Hz")
print("Array shape:", signal.shape, "| dtype:", signal.dtype)
print("Durasi yang dibaca:", round(len(signal) / sr_preview, 3), "detik")
print("10 sampel pertama:", signal[:10])


In [ ]:
try:
    info = sf.info(str(preview_path))
    print("Channels asli:", info.channels)
    print("Frames asli:", info.frames)
    print("Format / subtype:", info.format, "/", info.subtype)
    print("Durasi file penuh:", round(info.duration, 3), "detik")
except Exception:
    print("Metadata format tidak tersedia melalui soundfile; array preview tetap dapat dipakai.")


## 5. Waveform dan play audio
Sumbu waktu menggunakan detik. Amplitudo float dari librosa tidak sama skalanya dengan integer PCM mentah.


In [ ]:
time = np.arange(len(signal)) / sr_preview
plt.figure(figsize=(10, 4))
plt.plot(time, signal)
plt.xlabel("Waktu (detik)")
plt.ylabel("Amplitudo")
plt.title("Waveform audio sebelum preprocessing")
plt.grid(alpha=0.3)
plt.show()
display(Audio(signal, rate=sr_preview))


## 6. Spectrogram
Warna menunjukkan power spectral density dalam dB. Warna terang menunjukkan energi lebih tinggi pada waktu/frekuensi tersebut.


In [ ]:
nperseg = min(N_FFT, len(signal))
f, t, Sxx = spectrogram(signal, fs=sr_preview, nperseg=nperseg, noverlap=nperseg // 2)
plt.figure(figsize=(10, 4))
plt.pcolormesh(t, f, 10 * np.log10(Sxx + 1e-12), shading="auto")
plt.xlabel("Waktu (detik)")
plt.ylabel("Frekuensi (Hz)")
plt.title("Spectrogram audio")
plt.colorbar(label="PSD (dB)")
plt.tight_layout()
plt.show()


## 7. Preprocessing dan ekstraksi fitur

Setiap file → mono dan resampling → trim opsional → cek durasi → normalisasi/pre-emphasis
opsional → fitur rata-rata. File terlalu pendek di bawah `MIN_SECONDS` dibuang. Sinyal valid
yang lebih pendek daripada satu frame diberi zero padding dan ditandai `padded`.

13 MFCC + ZCR + RMS + centroid + bandwidth + rolloff = **18 fitur** pada pengaturan default.
Rata-rata merangkum satu file menjadi satu baris, sehingga detail urutan waktu tidak dipertahankan.


In [ ]:
features = []

for row in df_files.itertuples(index=False):
    try:
        y, sr = librosa.load(str(row.path), sr=TARGET_SR, mono=True, duration=MAX_SECONDS)
        if len(y) == 0 or not np.isfinite(y).all() or np.max(np.abs(y)) == 0:
            raise ValueError("Audio kosong, tidak finite, atau sepenuhnya senyap")
        if TRIM_SILENCE:
            y, _ = librosa.effects.trim(y, top_db=60)
        duration = len(y) / sr
        if duration < MIN_SECONDS:
            raise ValueError("Audio terlalu pendek setelah preprocessing")
        if NORMALIZE_AUDIO:
            y = librosa.util.normalize(y)
        padded = len(y) < N_FFT
        if padded:
            y = np.pad(y, (0, N_FFT - len(y)))
        if PREEMPHASIS:
            y = librosa.effects.preemphasis(y)

        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_mels=40,
                                   n_fft=N_FFT, hop_length=HOP_LENGTH)
        spectrum = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))
        feature_row = {"path": str(row.path), "label": row.label,
                       "duration_seconds": duration, "padded": padded,
                       "zcr": librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH).mean(),
                       "rms": librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH).mean(),
                       "spectral_centroid": librosa.feature.spectral_centroid(S=spectrum, sr=sr).mean(),
                       "spectral_bandwidth": librosa.feature.spectral_bandwidth(S=spectrum, sr=sr).mean(),
                       "spectral_rolloff": librosa.feature.spectral_rolloff(S=spectrum, sr=sr).mean()}
        for i in range(N_MFCC):
            feature_row[f"mfcc_{i+1}"] = mfcc[i].mean()
        features.append(feature_row)
    except Exception as error:
        failed.append({"path": str(row.path), "error": str(error)})

feature_columns = ["zcr", "rms", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff"]
feature_columns += [f"mfcc_{i+1}" for i in range(N_MFCC)]
df_features = pd.DataFrame(features, columns=["path", "label", "duration_seconds", "padded"] + feature_columns)


# STANDARDISASI BERSAMA — semua jalur lanjut dari sini

`df_features`, `feature_columns`, dan `failed` sudah tersedia: dari ekstraksi audio untuk
A/B/C/D atau langsung dari loader E. Pertahankan seluruh bagian setelah judul ini.


## 8. Cek fitur dan standardisasi
Hanya kolom fitur masuk scaler. Metadata durasi dan status padding disimpan sebagai informasi, bukan fitur clustering otomatis.


In [ ]:
if df_features.empty:
    display(pd.DataFrame(failed))
    raise ValueError("Tidak ada audio/fitur valid. Periksa file, codec, dan konfigurasi.")

valid = np.isfinite(df_features[feature_columns].to_numpy(dtype=float)).all(axis=1)
for path in df_features.loc[~valid, "path"]:
    failed.append({"path": str(path), "error": "Fitur mengandung NaN atau infinity"})
df_data = df_features.loc[valid].reset_index(drop=True)
df_failed = pd.DataFrame(failed, columns=["path", "error"])

print("Data valid:", len(df_data), "| Jumlah fitur:", len(feature_columns), "| Gagal:", len(df_failed))
display(df_data.head())
if not df_failed.empty:
    display(df_failed.head(10))
if len(df_data) < 3:
    raise ValueError("Perlu minimal 3 baris fitur valid.")


In [ ]:
X = df_data[feature_columns].to_numpy(dtype=float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Shape fitur standar:", X_scaled.shape)
print("Fitur konstan:", [name for name, std in zip(feature_columns, X.std(axis=0)) if std == 0])


## Clustering — pilih k atau ikuti k dari soal

Ubah `K_FIXED` di konfigurasi jika soal menentukan jumlah cluster. Jika `None`,
bandingkan `K_CANDIDATES` dan pilih silhouette tertinggi pada kandidat yang dicoba.
Ini pemilihan eksploratif pada data yang sama, bukan skor generalisasi test.

`X_model` adalah fitur yang dipakai K-Means. Silhouette dihitung pada ruang fitur ini,
bukan pada koordinat grafik dua dimensi. Untuk data besar, skor menggunakan maksimal
`SILHOUETTE_SAMPLE` baris acak yang sama pada setiap k. Skor sampel merupakan perkiraan.


In [ ]:
X_model = X_scaled

n_samples = X_model.shape[0]
if n_samples < 3:
    raise ValueError("Perlu minimal 3 data valid untuk alur clustering dan silhouette ini.")

k_values = [K_FIXED] if K_FIXED is not None else K_CANDIDATES
k_values = sorted(set(k for k in k_values if 2 <= k < n_samples))
if not k_values:
    raise ValueError("K harus minimal 2 dan lebih kecil dari jumlah data. Periksa konfigurasi.")

rng = np.random.default_rng(RANDOM_STATE)
eval_size = n_samples if SILHOUETTE_SAMPLE is None else min(SILHOUETTE_SAMPLE, n_samples)
if eval_size < 3:
    raise ValueError("SILHOUETTE_SAMPLE harus minimal 3, atau None.")
eval_index = rng.choice(n_samples, size=eval_size, replace=False)
X_eval = X_model[eval_index]
print("Data modeling:", X_model.shape, "| Data evaluasi silhouette:", eval_size)


In [ ]:
results = []
models = {}

for k in k_values:
    if USE_MINIBATCH:
        model = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                                n_init=10, batch_size=1024)
    else:
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)

    labels = model.fit_predict(X_model)
    eval_labels = labels[eval_index]
    score = np.nan

    if 2 <= len(np.unique(eval_labels)) < len(eval_labels):
        score = silhouette_score(X_eval, eval_labels, metric="euclidean")

    models[k] = model
    results.append({"k": k, "cluster_terbentuk": len(np.unique(labels)),
                    "silhouette": score, "inertia": model.inertia_})
    print(f"k={k} | silhouette={score:.4f} | inertia={model.inertia_:.2f}")

df_scores = pd.DataFrame(results)
display(df_scores)


`NaN` pada silhouette berarti sampel evaluasi tidak memenuhi syarat jumlah cluster.
Jika cluster sangat kecil terlewat oleh sampling, naikkan `SILHOUETTE_SAMPLE` atau
gunakan `None`. Data identik dapat menghasilkan lebih sedikit cluster daripada k.


In [ ]:
if K_FIXED is not None:
    best_k = K_FIXED
else:
    valid_scores = df_scores.dropna(subset=["silhouette"])
    if valid_scores.empty:
        raise ValueError("Tidak ada silhouette valid. Periksa data identik atau ukuran sampel evaluasi.")
    best_k = int(valid_scores.loc[valid_scores["silhouette"].idxmax(), "k"])

kmeans = models[best_k]
df_result = df_data.copy()
df_result["cluster"] = kmeans.predict(X_model)
print("K yang dipakai:", best_k)
display(df_result.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(df_scores["k"], df_scores["silhouette"], marker="o")
axes[0].set_title("Silhouette per k")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Silhouette")

axes[1].plot(df_scores["k"], df_scores["inertia"], marker="o")
axes[1].set_title("Elbow / inertia")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Inertia")
plt.tight_layout()
plt.show()


## Distribusi dan evaluasi tambahan

Silhouette mendekati 1 berarti pemisahan lebih baik; sekitar 0 berarti cluster
bertumpang tindih; nilai negatif menunjukkan sebagian data lebih dekat ke cluster lain.
Inertia biasanya turun ketika k naik; cari perubahan kemiringan untuk membantu membaca elbow.

ARI dan NMI hanya dihitung jika ada label acuan yang bermakna. Nomor cluster tidak
otomatis sama dengan label kelas. Jangan langsung memakai accuracy pada nomor cluster.


In [ ]:
cluster_count = df_result["cluster"].value_counts().sort_index()
display(cluster_count.rename("jumlah").to_frame())

cluster_count.plot(kind="bar", figsize=(7, 4))
plt.xlabel("Cluster")
plt.ylabel("Jumlah data")
plt.title("Distribusi cluster")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
label_mask = df_result["label"].notna()
if label_mask.sum() >= 2 and df_result.loc[label_mask, "label"].nunique() >= 2:
    y_true = df_result.loc[label_mask, "label"].astype(str)
    y_cluster = df_result.loc[label_mask, "cluster"]

    print("Jumlah data dengan label acuan:", len(y_true))
    print("ARI:", round(adjusted_rand_score(y_true, y_cluster), 4))
    print("NMI:", round(normalized_mutual_info_score(y_true, y_cluster), 4))
    display(pd.crosstab(y_true, y_cluster, rownames=["Label acuan"], colnames=["Cluster"]))
else:
    print("ARI/NMI dilewati: label acuan tidak tersedia atau hanya ada satu kelas.")


## Visualisasi PCA

PCA di sini hanya untuk visualisasi. K-Means dan silhouette tetap memakai `X_model`.
Explained variance menunjukkan berapa proporsi variasi fitur yang ditangkap grafik.
Titik yang berdekatan pada grafik belum tentu berdekatan pada semua dimensi asal.


In [ ]:
pca = PCA(n_components=min(2, X_model.shape[0], X_model.shape[1]), random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_model)
plot_y = X_pca[:, 1] if X_pca.shape[1] == 2 else np.zeros(len(X_pca))

plt.figure(figsize=(8, 6))
for cluster in sorted(df_result["cluster"].unique()):
    mask = df_result["cluster"].to_numpy() == cluster
    plt.scatter(X_pca[mask, 0], plot_y[mask], label=f"Cluster {cluster}", alpha=0.6, s=20)
plt.xlabel("PC1")
plt.ylabel("PC2" if X_pca.shape[1] == 2 else "Tidak ada komponen kedua")
plt.title("Visualisasi cluster dengan PCA")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print("Total explained variance:", round(pca.explained_variance_ratio_.sum(), 4))


## Interpretasi — profil fitur cluster

MFCC merangkum karakter spektral; ZCR menghitung perubahan tanda; RMS mewakili energi;
centroid adalah pusat massa spektrum; bandwidth menunjukkan sebaran spektrum; rolloff
adalah batas frekuensi yang mencakup proporsi energi tertentu (default librosa 85%).

Bandingkan fitur asli untuk interpretasi satuannya. Grafik memakai rata-rata fitur yang
sudah distandardisasi: nilai positif berarti di atas rata-rata seluruh data, negatif di bawah.
Cluster belum otomatis berarti normal/abnormal tanpa pemeriksaan isi dan label acuan.


In [ ]:
cluster_profile = df_result.groupby("cluster")[feature_columns].mean()
display(cluster_profile.round(3))

standardized_profile = pd.DataFrame(X_scaled, columns=feature_columns)
standardized_profile["cluster"] = df_result["cluster"].to_numpy()
standardized_profile = standardized_profile.groupby("cluster").mean()

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(standardized_profile.to_numpy(), cmap="coolwarm", aspect="auto", vmin=-2, vmax=2)
ax.set_xticks(range(len(feature_columns)), feature_columns, rotation=60, ha="right")
ax.set_yticks(range(len(standardized_profile)), standardized_profile.index)
ax.set_ylabel("Cluster")
ax.set_title("Rata-rata fitur standar per cluster")
fig.colorbar(im, ax=ax, label="Rata-rata nilai standar")
plt.tight_layout()
plt.show()


## Interpretasi — lokasi contoh audio tiap cluster
Buka atau dengarkan contoh anggota cluster jika diminta; untuk CSV fitur tanpa file asli, cukup gunakan profil fitur.


In [ ]:
for cluster in sorted(df_result["cluster"].unique()):
    print("Cluster", cluster)
    display(df_result.loc[df_result["cluster"] == cluster, ["path", "label"]].head(3))


## Simpan hasil (opsional)

Ubah `SAVE_RESULTS = True` jika ingin membuat CSV. Semua baris hasil mengikuti urutan
data yang benar-benar berhasil diproses. File gagal tersedia terpisah pada `df_failed`
jika notebook ini membaca kumpulan file.


In [ ]:
SAVE_RESULTS = False
OUTPUT_DIR = Path("hasil_uts")

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    df_result.to_csv(OUTPUT_DIR / "audio_hasil.csv", index=False)
    df_scores.to_csv(OUTPUT_DIR / "audio_skor_k.csv", index=False)
    print("Hasil disimpan di:", OUTPUT_DIR.resolve())


## Catatan saat UTS

- Durasi audio boleh berbeda karena setiap file diringkas menjadi fitur dengan panjang tetap.
- Jika waveform/play/spectrogram saja yang diminta, berhenti sebelum ekstraksi fitur.
- Untuk train/test: ekstraksi fitur dengan pengaturan sama, fit scaler dan K-Means pada train;
  pada test gunakan `scaler.transform(X_test)` dan `kmeans.predict(...)`.
- Jika codec gagal, gunakan WAV yang diberikan atau konversi dengan alat yang tersedia;
  mengganti nama ekstensi tidak mengubah format audio.

- Jika soal menetapkan k, ubah `K_FIXED`, lalu jalankan kembali bagian clustering dan hasil setelahnya.
- Jika konfigurasi input atau preprocessing berubah, jalankan ulang dari konfigurasi sampai akhir.
- `MAX_ITEMS = None` berarti memakai seluruh data; angka tertentu berarti sampel acak yang dilaporkan.
- Untuk data besar, coba `USE_MINIBATCH = True` bila soal mengizinkan MiniBatch K-Means;
  hasilnya dapat berbeda dari K-Means biasa. Silhouette sampling hanya mempercepat evaluasi,
  sedangkan `MAX_ITEMS` atau MiniBatch membantu mengurangi beban modeling.
- Sesuaikan metode dengan instruksi soal. Template ini berfokus pada clustering seperti notebook contoh.
- Hasil k terbaik hanya terbaik di antara kandidat dan representasi yang dicoba; tidak menjamin cluster
  tersebut paling bermakna. Jelaskan juga isi cluster dan keterbatasan fitur.


Referensi API: [librosa.load](https://librosa.org/doc/0.11.0/generated/librosa.load.html),
[fitur librosa](https://librosa.org/doc/0.11.0/feature.html),
[silhouette_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html).

Kaggle: [dokumentasi resmi kagglehub](https://github.com/Kaggle/kagglehub).
